## Imports


In [1]:
from pathlib import Path
from collections import Counter
import json
import re
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits

## Configuration


In [2]:
RANDOM_STATE = 42
C_SVM = 20.0
POIDS_SVM = 0.90
FAMILLES = ['contextes', 'actions', 'sequences', 'parcours']
POIDS_FAMILLES = [1.0, 1.0, 1.0, 0.5]
DATA_DIR = None
if DATA_DIR is None:
    candidats = [p / 'data' for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'data/train.csv').is_file()]
    if not candidats and Path('/kaggle/input').exists():
        candidats = [p.parent for p in Path('/kaggle/input').rglob('train.csv')
                     if (p.parent / 'test.csv').is_file()]
    assert len(candidats) == 1, 'Renseigner DATA_DIR avec le dossier des données.'
    DATA_DIR = candidats[0]
DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR.parent / 'datamanip'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'C': C_SVM, 'poids_svm': POIDS_SVM, 'poids_extra_trees': round(1 - POIDS_SVM, 2)})

{'C': 20.0, 'poids_svm': 0.9, 'poids_extra_trees': 0.1}


## Données


In [3]:
def charger_sessions(path, etiquete):
    lignes = []
    with path.open(encoding='utf-8-sig') as fichier:
        for ligne in fichier:
            champs = ligne.rstrip('\r\n').split(',')
            assert len(champs) >= (2 if etiquete else 1)
            lignes.append({'util': champs[0] if etiquete else None,
                           'navigateur': champs[1] if etiquete else champs[0],
                           'traces': champs[2:] if etiquete else champs[1:]})
    return pd.DataFrame(lignes)

train_df = charger_sessions(DATA_DIR / 'train.csv', True)
test_df = charger_sessions(DATA_DIR / 'test.csv', False)
soumission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
assert list(soumission.columns) == ['RowId', 'prediction']
assert len(soumission) == len(test_df)
assert soumission.RowId.is_unique
assert np.array_equal(soumission.RowId.to_numpy(), np.arange(1, len(test_df) + 1))
df = pd.concat([train_df, test_df], ignore_index=True)
y = train_df.util.to_numpy()
classes = np.unique(y)
navigateurs = df.navigateur.to_numpy()
indices_train = np.arange(len(train_df))
indices_test = np.arange(len(train_df), len(df))
print({'train': len(train_df), 'test': len(test_df), 'utilisateurs': len(classes)})

{'train': 3279, 'test': 324, 'utilisateurs': 247}


## Préparation


In [4]:
def est_temps(trace):
    return re.fullmatch('t\\d+', trace) is not None

def type_action(action):
    nom = re.split('[(<$]', action, maxsplit=1)[0].strip()
    return nom[:-1].rstrip() if nom.endswith('1') else nom

def contextes(actions, motif):
    return [valeur for action in actions for valeur in re.findall(motif, action)]

def ajouter_comptages(cible, prefixe, valeurs):
    for valeur, nombre in Counter(valeurs).items():
        cible[prefixe + json.dumps(valeur, ensure_ascii=False)] = nombre

def ngrammes(valeurs, taille):
    return zip(*(valeurs[j:] for j in range(taille)))

df['actions'] = df.traces.map(lambda traces: [t for t in traces if not est_temps(t)])

## Variables des arbres


In [5]:
def caracteristiques_reference(ligne):
    actions = ligne['actions']
    verbes = [type_action(a) for a in actions]
    temps = [int(t[1:]) for t in ligne['traces'] if est_temps(t)]
    ecrans = contextes(actions, '\\((.*?)\\)')
    configs = contextes(actions, '<(.*?)>')
    chaines = contextes(actions, '\\$(.*?)\\$')
    n = len(actions)
    modifications = sum((a.endswith('1') for a in actions))
    resultat = dict(nb_actions=n, nb_marqueurs_temporels=len(temps), dernier_repere_s=max(temps, default=0), sans_actions=int(n == 0), sans_marqueur_temporel=int(not temps), nb_actions_modification=modifications, part_actions_modification=modifications / n if n else 0, nb_types_actions_distincts=len(set(verbes)), nb_ecrans_contextes_distincts=len(set(ecrans)), nb_configurations_distinctes=len(set(configs)), nb_chaines_distinctes=len(set(chaines)), navigateur=ligne['navigateur'])
    for prefixe, valeurs in [('action_nombre', verbes), ('ecran_nombre', ecrans), ('configuration_nombre', configs), ('chaine_nombre', chaines)]:
        for valeur, nombre in Counter(valeurs).items():
            resultat[f'{prefixe}::{valeur}'] = nombre
    for valeur, nombre in Counter(verbes).items():
        resultat[f'action_proportion::{valeur}'] = nombre / n
    return resultat

X_reference = [caracteristiques_reference(r) for r in df.to_dict('records')]

## Variables des séquences


In [6]:
def representations(actions, navigateur):
    verbes = [type_action(a) for a in actions]
    ecrans = contextes(actions, '\\((.*?)\\)')
    configurations = contextes(actions, '<(.*?)>')
    chaines = contextes(actions, '\\$(.*?)\\$')
    contexte = {'navigateur': navigateur}
    for prefixe, valeurs in [('action_nombre', verbes), ('ecran_nombre', ecrans), ('configuration_nombre', configurations), ('chaine_nombre', chaines)]:
        for valeur, nombre in Counter(valeurs).items():
            contexte[f'{prefixe}::{valeur}'] = nombre
    complet, sequences, parcours = ({}, {}, {})
    ajouter_comptages(complet, 'complete::', actions)
    for taille in (2, 3):
        ajouter_comptages(sequences, f'complete{taille}::', ngrammes(actions, taille))
        ajouter_comptages(parcours, f'verbe{taille}::', ngrammes(verbes, taille))
    ajouter_comptages(parcours, 'transition::', ngrammes(ecrans, 2))
    return {'historique': {**contexte, **complet, **sequences, **parcours}, 'contextes': contexte, 'actions': complet, 'sequences': sequences, 'parcours': parcours}

REPRESENTATIONS = [representations(a, n) for a, n in zip(df.actions, navigateurs)]

## TF-IDF par blocs


In [7]:
def encoder_blocs(train, evaluation):
    blocs_train, blocs_eval = [], []
    for famille, poids_bloc in zip(FAMILLES, POIDS_FAMILLES):
        vecteur = DictVectorizer(dtype=np.float32)
        a = vecteur.fit_transform([REPRESENTATIONS[i][famille] for i in train])
        b = vecteur.transform([REPRESENTATIONS[i][famille] for i in evaluation])
        garder = np.asarray((a != 0).sum(axis=0)).ravel() >= 2
        a, b = a[:, garder], b[:, garder]
        tfidf = TfidfTransformer(sublinear_tf=True)
        blocs_train.append(poids_bloc * tfidf.fit_transform(a))
        blocs_eval.append(poids_bloc * tfidf.transform(b))
    return (normalize(sparse.hstack(blocs_train, format='csr')),
            normalize(sparse.hstack(blocs_eval, format='csr')))

## Extra Trees


In [8]:
def predire_arbres(train, evaluation):
    resultat = np.zeros((len(evaluation), len(classes)), dtype=np.float32)
    for navigateur in np.unique(navigateurs[evaluation]):
        tr = train[navigateurs[train] == navigateur]
        positions = np.flatnonzero(navigateurs[evaluation] == navigateur)
        ev = evaluation[positions]
        specialise = len(tr) > 0
        if not specialise:
            tr = train
        if len(np.unique(y[tr])) == 1:
            resultat[positions, np.searchsorted(classes, y[tr[0]])] = 1
            continue

        def lignes(indices):
            return [{k: v for k, v in X_reference[i].items() if not (specialise and k == 'navigateur')} for i in indices]
        vecteur = DictVectorizer(dtype=np.float32)
        a = vecteur.fit_transform(lignes(tr))
        b = vecteur.transform(lignes(ev))
        modele = ExtraTreesClassifier(n_estimators=200, max_features=0.5, class_weight='balanced', n_jobs=2, random_state=RANDOM_STATE)
        with threadpool_limits(limits=2):
            modele.fit(a, y[tr])
            proba = modele.predict_proba(b)
        resultat[np.ix_(positions, np.searchsorted(classes, modele.classes_))] = proba
    return resultat

## SVM


In [9]:
def filtrer_navigateur(scores, train, evaluation):
    resultat = scores.copy()
    for navigateur in np.unique(navigateurs[evaluation]):
        autorises = np.unique(y[train[navigateurs[train] == navigateur]])
        if len(autorises):
            positions = np.flatnonzero(navigateurs[evaluation] == navigateur)
            interdits = np.flatnonzero(~np.isin(classes, autorises))
            resultat[np.ix_(positions, interdits)] = 0
    return resultat / np.maximum(resultat.sum(axis=1, keepdims=True), 1e-30)

def predire_svm(train, evaluation):
    a, b = encoder_blocs(train, evaluation)
    modele = LinearSVC(C=C_SVM, dual=True, max_iter=30000, random_state=RANDOM_STATE)
    with warnings.catch_warnings(), threadpool_limits(limits=2):
        warnings.simplefilter('error', ConvergenceWarning)
        modele.fit(a, y[train], sample_weight=np.ones(len(train)))
        marges = modele.decision_function(b)
    resultat = np.zeros((len(evaluation), len(classes)), dtype=np.float32)
    resultat[:, np.searchsorted(classes, modele.classes_)] = softmax(3 * marges, axis=1)
    return filtrer_navigateur(resultat, train, evaluation)

## Ensemble


In [10]:
def predire_ensemble(train, evaluation):
    svm = predire_svm(train, evaluation)
    arbres = predire_arbres(train, evaluation)
    scores = POIDS_SVM * svm + (1 - POIDS_SVM) * arbres
    assert np.isfinite(scores).all()
    assert np.allclose(scores.sum(axis=1), 1, atol=1e-5)
    return classes[scores.argmax(axis=1)]

## Validation de développement


In [11]:
apprentissage, validation = train_test_split(
    indices_train, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
prediction_validation = predire_ensemble(apprentissage, validation)
resultats = pd.DataFrame([{
    'F1 macro': f1_score(y[validation], prediction_validation, average='macro', zero_division=0),
    'F1 pondéré': f1_score(y[validation], prediction_validation, average='weighted', zero_division=0),
    'Accuracy': accuracy_score(y[validation], prediction_validation),
    'Erreurs': int(np.sum(y[validation] != prediction_validation))
}])
print(resultats.to_string(index=False))

 F1 macro  F1 pondéré  Accuracy  Erreurs
 0.952907    0.956579  0.960366       26


## Entraînement complet


In [12]:
predictions_test = predire_ensemble(indices_train, indices_test)
assert len(predictions_test) == len(test_df)
assert np.isin(predictions_test, classes).all()
print({'sessions_apprentissage': len(indices_train), 'predictions': len(predictions_test)})

{'sessions_apprentissage': 3279, 'predictions': 324}


## Soumission Kaggle


In [13]:
soumission['prediction'] = predictions_test
assert not soumission.isna().any().any()
fichier_soumission = OUTPUT_DIR / 'submission.csv'
soumission.to_csv(fichier_soumission, index=False)
verification = pd.read_csv(fichier_soumission)
pd.testing.assert_frame_equal(verification, soumission)
print(soumission.head().to_string(index=False))
print(fichier_soumission.resolve())

 RowId prediction
     1        hzx
     2        prb
     3        jmq
     4        psg
     5        cvw
C:\Users\Rog strix Scar\Desktop\BE_MachineLearning\datamanip\submission.csv
